# FineBadminton frame cache â€” FBBench

Extract frames **once**, on CPU, so GPU quota is never spent decoding video.

FBBench ships its own clips (`hf_video_clips_qa/video_NNNNNN.mp4`, one per
question, named by `video_uid`), so this kernel needs only the benchmark repo,
not Finebadminton-20K. See `results/dataset_report.md`.

With `LIMIT` set, only that many clips are downloaded, so a test run takes
minutes. `LIMIT = None` downloads all 10.75 GB of clips.

Output (`/kaggle/working`): `frames/<video_uid>/<index:03d>_<ms:07d>.jpg`,
`frames/manifest.json`, and `fbbench/` holding the question JSONs only.
Attach this kernel's output to the GPU notebook via `kernel_sources`.

In [ ]:
# 1. Config
FPS = 2          # matches the FineBadminton uniform baseline (~25 frames / rally)
SIZE = 448       # short-side pixels; downscale at load time, never upscale
LIMIT = None     # e.g. 20 for a quick test run
WORKERS = 4
JPEG_QUALITY = 88

REPO_ID = "Moujuruo/FineBadmintonBenchmark"
REVISION = "ff86093a51efdf9a243a0635704952d404e99e52"   # pinned; see dataset_report.md
CLIP_DIR_IN_REPO = "hf_video_clips_qa"

import json
import pathlib
import shutil
import subprocess

TEMP = pathlib.Path("/kaggle/temp")
FBBENCH_DIR = TEMP / "fbbench"
CLIPS_DIR = FBBENCH_DIR / CLIP_DIR_IN_REPO
WORK = pathlib.Path("/kaggle/working")
OUT = WORK / "frames"

In [ ]:
# 2. Download the question JSONs, read the video_uids, then download only those clips
from huggingface_hub import snapshot_download

subprocess.run(["pip", "install", "-q", "decord"], check=True)
TEMP.mkdir(parents=True, exist_ok=True)

snapshot_download(REPO_ID, repo_type="dataset", revision=REVISION, local_dir=str(FBBENCH_DIR),
                  allow_patterns=["finebadmintonbenchmark/benchmark_*.json", "README.md"])

bench_files = sorted(FBBENCH_DIR.glob("finebadmintonbenchmark/benchmark_*.json"))
records = {p.name: json.loads(p.read_text(encoding="utf-8")) for p in bench_files}
wanted = sorted({r["video_uid"] for recs in records.values() for r in recs})
print(f"{len(bench_files)} benchmark files, {sum(map(len, records.values()))} questions, "
      f"{len(wanted)} distinct video_uids")

selected = wanted[:LIMIT] if LIMIT else wanted
patterns = ([f"{CLIP_DIR_IN_REPO}/{u}.mp4" for u in selected] if LIMIT
            else [f"{CLIP_DIR_IN_REPO}/*.mp4"])
snapshot_download(REPO_ID, repo_type="dataset", revision=REVISION, local_dir=str(FBBENCH_DIR),
                  allow_patterns=patterns)

subprocess.run(["du", "-sh", str(FBBENCH_DIR), str(CLIPS_DIR)])

In [ ]:
# 3. Match video_uids to clips on disk
VIDEO_EXT = {".mp4", ".avi", ".mkv", ".mov", ".webm"}

have = {p.stem: p for p in CLIPS_DIR.rglob("*") if p.suffix.lower() in VIDEO_EXT}
matched = [u for u in selected if u in have]
missing = [u for u in selected if u not in have]

print(f"selected for this run: {len(selected)}")
print(f"videos on disk:        {len(have)}")
print(f"matched:               {len(matched)}")
if missing:
    print(f"MISSING, e.g.:         {missing[:5]}")
if not matched:
    first = next(iter(records.values()))[0]
    print("\nZERO MATCHED. One full record:")
    print(json.dumps(first, indent=2, ensure_ascii=False))
    print("Sample files on disk:", [p.name for p in list(CLIPS_DIR.rglob('*'))[:5]])
    raise SystemExit("no clips matched -- fix before extracting")

In [ ]:
# 4. Extract frames with decord, WORKERS processes
import multiprocessing as mp
import statistics
from concurrent.futures import ProcessPoolExecutor, as_completed


def extract_one(video: pathlib.Path, out_dir: pathlib.Path,
                fps: float, size: int, quality: int) -> dict:
    """Same logic as scripts/extract_frames.py; returns one manifest row."""
    clip_id = video.stem
    dest = out_dir / clip_id

    if dest.exists() and any(dest.glob("*.jpg")):
        return {"clip_id": clip_id, "frames": len(list(dest.glob("*.jpg"))), "status": "skipped"}

    try:
        import decord
        from PIL import Image

        vr = decord.VideoReader(str(video), num_threads=1)
        src_fps = float(vr.get_avg_fps()) or 25.0
        step = max(1, round(src_fps / fps))
        idx = list(range(0, len(vr), step))
        if not idx:
            return {"clip_id": clip_id, "frames": 0, "status": "empty"}

        batch = vr.get_batch(idx).asnumpy()
        dest.mkdir(parents=True, exist_ok=True)
        h0, w0 = batch.shape[1:3]

        for n, (frame, src_i) in enumerate(zip(batch, idx)):
            im = Image.fromarray(frame)
            w, h = im.size
            scale = size / min(w, h)                      # short side -> `size`
            im = im.resize((round(w * scale), round(h * scale)), Image.BICUBIC)
            # timestamp in the filename keeps the cache self-describing
            t_ms = round(src_i / src_fps * 1000)
            im.save(dest / f"{n:03d}_{t_ms:07d}.jpg", quality=quality)

        return {"clip_id": clip_id, "frames": len(idx), "status": "ok",
                "src_fps": round(src_fps, 3), "src_frames": len(vr),
                "duration_s": round(len(vr) / src_fps, 2), "src_wh": [int(w0), int(h0)]}

    except Exception as e:                                 # noqa: BLE001
        return {"clip_id": clip_id, "frames": 0, "status": f"error: {type(e).__name__}: {e}"}


todo = [have[u] for u in matched]
print(f"extracting {len(todo)} clips at {FPS} fps, short side {SIZE}px")

OUT.mkdir(parents=True, exist_ok=True)
results = []

# fork, explicitly: functions defined in a notebook cell cannot be pickled
# by reference under spawn/forkserver
with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context("fork")) as pool:
    futs = [pool.submit(extract_one, v, OUT, FPS, SIZE, JPEG_QUALITY) for v in todo]
    for i, fut in enumerate(as_completed(futs), 1):
        results.append(fut.result())
        if i % 100 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)}")

results.sort(key=lambda r: r["clip_id"])
errors = [r for r in results if r["status"].startswith("error") or r["status"] == "empty"]
(OUT / "manifest.json").write_text(json.dumps({
    "source": REPO_ID, "revision": REVISION,
    "fps": FPS, "size": SIZE, "jpeg_quality": JPEG_QUALITY, "limit": LIMIT,
    "clips": len(results),
    "frames": sum(r["frames"] for r in results),
    "missing_videos": missing,
    "results": results,
}, indent=1))

counts = [r["frames"] for r in results if r["frames"]]
durs = [r["duration_s"] for r in results if "duration_s" in r]
mb = sum(f.stat().st_size for f in OUT.rglob("*.jpg")) / 1e6
print(f"\ndone: {len(results)} clips, {sum(counts)} frames, {mb:.0f} MB")
if counts:
    print(f"frames/clip   min {min(counts)}  median {statistics.median(counts)}  max {max(counts)}")
if durs:
    print(f"clip seconds  min {min(durs)}  median {statistics.median(durs)}  max {max(durs)}")
    print("source fps    ", sorted({r["src_fps"] for r in results if "src_fps" in r}))
    print("source WxH    ", sorted({tuple(r["src_wh"]) for r in results if "src_wh" in r}))
if errors:
    print(f"{len(errors)} FAILED, first few: {errors[:3]}")

In [ ]:
# 5. Keep only the question JSONs with the output, drop the clips, report disk usage
dest = WORK / "fbbench" / "finebadmintonbenchmark"
dest.mkdir(parents=True, exist_ok=True)
for p in bench_files:
    shutil.copy2(p, dest / p.name)
shutil.copy2(FBBENCH_DIR / "README.md", WORK / "fbbench" / "README.md")
shutil.rmtree(CLIPS_DIR, ignore_errors=True)

subprocess.run(["du", "-sh", str(OUT), str(WORK / "fbbench"), str(WORK)])
subprocess.run(["df", "-h", str(WORK), str(TEMP)])